In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [49]:
ratings = pd.read_csv('BX-Book-Ratings.csv', encoding='latin-1', nrows=100000)
print(ratings.head())
books = pd.read_csv('BX-Books.csv', encoding='latin-1', nrows=100000)
print(books.head())
users = pd.read_csv('BX-Users.csv', encoding='latin-1', nrows=100000)
print(users.head())

   user_id        isbn  rating
0   276725  034545104X       0
1   276726   155061224       5
2   276727   446520802       0
3   276729  052165615X       3
4   276729   521795028       6
        isbn                                         book_title  \
0  195153448                                Classical Mythology   
1    2005018                                       Clara Callan   
2   60973129                               Decision in Normandy   
3  374157065  Flu: The Story of the Great Influenza Pandemic...   
4  393045218                             The Mummies of Urumchi   

            book_author year_of_publication                   publisher  
0    Mark P. O. Morford                2002     Oxford University Press  
1  Richard Bruce Wright                2001       HarperFlamingo Canada  
2          Carlo D'Este                1991             HarperPerennial  
3      Gina Bari Kolata                1999        Farrar Straus Giroux  
4       E. J. W. Barber                19

In [50]:
x = ratings['user_id'].value_counts() > 200
y = x[x].index  #user_ids
ratings = ratings[ratings['user_id'].isin(y)]
ratings.head()

,user_id,isbn,rating
1456,277427,002542730X,10
1457,277427,26217457,0
1458,277427,003008685X,8
1459,277427,30615321,0
1460,277427,60002050,0


In [56]:
rating_with_books = ratings.merge(books, on='isbn')
number_rating = rating_with_books.groupby('book_title')['rating'].count().reset_index()
number_rating.rename(columns= {'rating':'number_of_ratings'}, inplace=True)
final_rating = rating_with_books.merge(number_rating, on='book_title')

In [58]:
book_pivot = final_rating.pivot_table(columns='user_id', index='book_title', values="rating")
book_pivot.fillna(0, inplace=True)

In [68]:
from scipy.sparse import csr_matrix
book_sparse = csr_matrix(book_pivot)
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(algorithm='brute')
model.fit(book_sparse)
distances, suggestions = model.kneighbors(book_pivot.iloc[230, :].values.reshape(1, -1))
print("Suggestions for the book - ", book_pivot.iloc[230].name)
for i in range(len(suggestions)):
    print(book_pivot.index[suggestions[i]])

Suggestions for the book -  A Civil Action
Index(['A Civil Action', 'Wuthering Heights', 'The Thorn Birds',
       'Silent Treatment', 'Interview with the Vampire'],
      dtype='object', name='book_title')
